In [1]:
import pandas as pd

matches = pd.read_csv("data/t20i_matches.csv")
balls = pd.read_csv("data/t20i_balls.csv")

print(matches.head())
print(balls.head())

   match_id      team1         team2     winner   toss_winner toss_decision  \
0   1475527  Hong Kong      Malaysia  Hong Kong     Hong Kong           bat   
1   1359789    Bermuda        Panama    Bermuda       Bermuda           bat   
2   1515547   Cambodia     Indonesia  Indonesia     Indonesia         field   
3   1400974    Nigeria  Sierra Leone    Nigeria  Sierra Leone         field   
4   1501504   Eswatini    Mozambique   Eswatini      Eswatini           bat   

                                         venue          city        date  \
0                  Bayuemas Oval, Kuala Lumpur  Kuala Lumpur  2025-03-12   
1  Belgrano Athletic Club Ground, Buenos Aires  Buenos Aires  2023-02-28   
2                       Udayana Cricket Ground          Bali  2025-12-27   
3     Tafawa Balewa Square Cricket Oval, Lagos         Lagos  2023-10-05   
4                   Malkerns Country Club oval      Malkerns  2025-09-13   

  gender  
0   male  
1   male  
2   male  
3   male  
4   male  
  

In [2]:
import pandas as pd
import numpy as np

matches_df = pd.read_csv("data/t20i_matches.csv")
balls_df = pd.read_csv("data/t20i_balls.csv")

# Ensure types
matches_df["match_id"] = matches_df["match_id"].astype(str)
balls_df["match_id"] = balls_df["match_id"].astype(str)

# Bring team1/team2 onto balls
balls = balls_df.merge(
    matches_df[["match_id", "team1", "team2"]],
    on="match_id",
    how="left"
)

# Infer bowling_team
balls["bowling_team"] = np.where(
    balls["batting_team"] == balls["team1"],
    balls["team2"],
    balls["team1"]
)

# Sort key inside innings
balls["ball_innings_index"] = balls["over"] * 6 + (balls["ball"] - 1)

balls = balls.sort_values(["match_id", "innings", "ball_innings_index"]).reset_index(drop=True)

# Innings cumulative runs and wickets
balls["runs_so_far"] = balls.groupby(["match_id", "innings"])["total_runs"].cumsum()
balls["wickets_lost"] = balls.groupby(["match_id", "innings"])["wicket"].cumsum()

# Simple momentum: runs in last 12 balls (based on your current ball indexing)
# (Later you can switch to legal-ball based rolling if you parse wide/noball flags.)
balls["runs_last_12"] = (
    balls.groupby(["match_id", "innings"])["total_runs"]
         .rolling(12, min_periods=1)
         .sum()
         .reset_index(level=[0,1], drop=True)
)

# Save processed
balls.to_csv("data/balls_with_state_basic.csv", index=False)

print(balls[[
    "match_id","innings","over","ball","batting_team","bowling_team",
    "total_runs","wicket","runs_so_far","wickets_lost","runs_last_12"
]].head(20))

   match_id  innings  over  ball batting_team bowling_team  total_runs  \
0   1001349        1     0     1    Australia    Sri Lanka           0   
1   1001349        1     0     2    Australia    Sri Lanka           0   
2   1001349        1     0     3    Australia    Sri Lanka           1   
3   1001349        1     0     4    Australia    Sri Lanka           2   
4   1001349        1     0     5    Australia    Sri Lanka           0   
5   1001349        1     0     6    Australia    Sri Lanka           3   
6   1001349        1     1     1    Australia    Sri Lanka           0   
7   1001349        1     1     2    Australia    Sri Lanka           1   
8   1001349        1     1     3    Australia    Sri Lanka           0   
9   1001349        1     1     4    Australia    Sri Lanka           0   
10  1001349        1     1     5    Australia    Sri Lanka           4   
11  1001349        1     1     6    Australia    Sri Lanka           2   
12  1001349        1     2     1    Au

In [5]:
assert balls["team1"].notna().all()
assert balls["team2"].notna().all()

In [6]:
innings1_final = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["runs_so_far"]
    .max()
)
print(innings1_final.describe())

count    1169.000000
mean      167.021386
std        32.948414
min        56.000000
25%       147.000000
50%       167.000000
75%       188.000000
max       287.000000
Name: runs_so_far, dtype: float64


In [7]:
assert (balls["wickets_lost"] <= 10).all()

In [9]:
import pandas as pd

balls = pd.read_csv("data/balls_with_state_basic.csv")

# Get innings1 totals
innings1_totals = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["runs_so_far"]
    .max()
    .reset_index()
)

innings1_totals["target"] = innings1_totals["runs_so_far"] + 1
innings1_totals = innings1_totals[["match_id", "target"]]

# Merge target into balls table
balls = balls.merge(innings1_totals, on="match_id", how="left")

# Balls used in innings
balls["balls_used"] = balls["over"] * 6 + balls["ball"]

# Balls remaining (T20 = 120 balls)
balls["balls_remaining"] = 120 - balls["balls_used"]

# Wickets remaining
balls["wickets_remaining"] = 10 - balls["wickets_lost"]

# Runs needed (only meaningful for innings 2)
balls["runs_needed"] = balls["target"] - balls["runs_so_far"]

# Required run rate
balls["required_rr"] = balls["runs_needed"] / (balls["balls_remaining"] / 6)

# Clean impossible values
balls.loc[balls["innings"] == 1, ["runs_needed", "required_rr"]] = None

print(balls.head(20))

    match_id  innings  over  ball         batting_team       batsman  \
0    1082591        1     0     1  Sunrisers Hyderabad     DA Warner   
1    1082591        1     0     2  Sunrisers Hyderabad     DA Warner   
2    1082591        1     0     3  Sunrisers Hyderabad     DA Warner   
3    1082591        1     0     4  Sunrisers Hyderabad     DA Warner   
4    1082591        1     0     5  Sunrisers Hyderabad     DA Warner   
5    1082591        1     0     6  Sunrisers Hyderabad      S Dhawan   
6    1082591        1     0     7  Sunrisers Hyderabad      S Dhawan   
7    1082591        1     1     1  Sunrisers Hyderabad      S Dhawan   
8    1082591        1     1     2  Sunrisers Hyderabad     DA Warner   
9    1082591        1     1     3  Sunrisers Hyderabad     DA Warner   
10   1082591        1     1     4  Sunrisers Hyderabad     DA Warner   
11   1082591        1     1     5  Sunrisers Hyderabad     DA Warner   
12   1082591        1     1     6  Sunrisers Hyderabad  MC Henri

In [10]:
balls[balls["innings"] == 2][[
    "runs_so_far",
    "target",
    "runs_needed",
    "balls_remaining",
    "required_rr"
]].head(10)

,runs_so_far,target,runs_needed,balls_remaining,required_rr
125,1,208,207.0,119,10.436975
126,1,208,207.0,118,10.525424
127,1,208,207.0,117,10.615385
128,3,208,205.0,116,10.603448
129,7,208,201.0,115,10.486957
130,11,208,197.0,114,10.368421
131,11,208,197.0,113,10.460177
132,11,208,197.0,112,10.553571
133,12,208,196.0,111,10.594595
134,12,208,196.0,110,10.690909


In [3]:
import pandas as pd

# ── Team lists ────────────────────────────────────────────────────────────────

TOP_9 = {
    "Australia", "England", "India", "Pakistan", "South Africa",
    "New Zealand", "Sri Lanka", "West Indies", "Bangladesh"
}

TIER_18 = TOP_9 | {
    "Afghanistan", "Ireland", "Zimbabwe", "Netherlands", "Scotland",
    "Namibia", "UAE", "Nepal", "Oman"
}

# ── Load data ─────────────────────────────────────────────────────────────────

matches = pd.read_csv("data/t20i_matches.csv", dtype={"match_id": str})
balls   = pd.read_csv("data/t20i_balls.csv",   dtype={"match_id": str})

print(f"Before filtering — matches: {len(matches):,}  |  balls: {len(balls):,}")

# ── Apply inclusion rules ─────────────────────────────────────────────────────
# Include a match if EITHER:
#   Rule 1: both teams are in the 18-team list
#   Rule 2: at least one team is a top-9 nation (opponent can be anyone)

def include_match(team1, team2):
    both_in_18    = (team1 in TIER_18) and (team2 in TIER_18)
    top9_involved = (team1 in TOP_9)   or  (team2 in TOP_9)
    return both_in_18 or top9_involved

mask = matches.apply(lambda r: include_match(r["team1"], r["team2"]), axis=1)

matches_filtered = matches[mask].copy()
balls_filtered   = balls[balls["match_id"].isin(matches_filtered["match_id"])].copy()

print(f"After filtering  — matches: {len(matches_filtered):,}  |  balls: {len(balls_filtered):,}")
print(f"Matches removed  : {len(matches) - len(matches_filtered):,}")

# ── Teams present after filtering ────────────────────────────────────────────
all_teams = pd.concat([
    matches_filtered["team1"],
    matches_filtered["team2"]
]).value_counts()

print(f"\nUnique teams remaining: {len(all_teams)}")
print("\nMatch count per team:")
print(all_teams.to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
matches_filtered.to_csv("data/t20i_matches_filtered.csv", index=False)
balls_filtered.to_csv("data/t20i_balls_filtered.csv",     index=False)

print("\nSaved → data/t20i_matches_filtered.csv")
print("Saved → data/t20i_balls_filtered.csv")

Before filtering — matches: 3,201  |  balls: 722,509
After filtering  — matches: 1,359  |  balls: 311,519
Matches removed  : 1,842

Unique teams remaining: 27

Match count per team:
Pakistan                    285
India                       266
New Zealand                 256
West Indies                 241
Australia                   224
England                     222
South Africa                221
Sri Lanka                   220
Bangladesh                  192
Zimbabwe                    146
Ireland                     109
Netherlands                  85
Scotland                     63
Namibia                      48
Oman                         43
Nepal                        41
United Arab Emirates         20
United States of America     10
Hong Kong                     6
Kenya                         4
ICC World XI                  4
Canada                        3
Papua New Guinea              3
Uganda                        2
Italy                         2
Ghana             